# colab_00 — Setup + environment capture

First notebook of the `ad-glia-fm-prep` execution phase. Mounts Drive, clones the repo, installs `requirements_integration.txt`, captures the resolved software stack into `outputs/software_versions/`, and runs preemptive Audits F (storage budget) and B (APOE metadata presence) before any count-matrix download.

**Order of operations:**

1. Mount Drive (section 1)
2. Clone/pull repo + install env (section 2)
3. Env capture — pip freeze + Python/CUDA/GPU/OS (section 3)
4. Audit F — storage budget table (section 4)
5. Audit B preemptive — donor metadata only, confirm APOE column (section 5)
6. Handoff to `colab_01_li2025_qc.ipynb` (section 6)

No bulk data download in this notebook beyond per-study donor-metadata files (typically <10 MB).

## 1 — Mount Google Drive

Drive persists across runtime resets; later notebooks write large intermediates (h5ad, FM caches, embeddings) here. `colab_00` doesn't produce large artifacts itself, but mounting + confirming the project root exists belongs to setup.

### 1a — Mount Drive and confirm project root

Mounts `/content/drive` and ensures `MyDrive/ad-glia-fm-prep/` exists. This is the Drive-side mirror for large data — runtime-local repo at `/content/ad-glia-fm-prep` is separate (committable artifacts live there).

In [5]:
import os
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f"DRIVE_ROOT = {DRIVE_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_ROOT = /content/drive/MyDrive/ad-glia-fm-prep


**What this shows.** Drive is mounted and `DRIVE_ROOT = /content/drive/MyDrive/ad-glia-fm-prep` is confirmed. "Drive already mounted" is not an error — it just means a mount already existed earlier in this session. colab_00 writes nothing to Drive; the mount matters for later notebooks that stage large h5ad files and FM caches here.

## 2 — Clone repo and install integration env

Repo is cloned to runtime-local `/content/ad-glia-fm-prep` (git ops are slow over the Drive FUSE mount). Committable artifacts (`outputs/audit_report.json`, `outputs/software_versions/*`) are written under the repo path so a final `git add + commit + push` from within this session lands them on GitHub.

### 2a — Clone or pull repo

Clones on first run, pulls otherwise. Adds the repo to `sys.path` so `from src.* import ...` works.

In [6]:
import os, subprocess, sys

REPO_URL = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

subprocess.run(["git", "-C", REPO_PATH, "rev-parse", "--short", "HEAD"], check=True)

CompletedProcess(args=['git', '-C', '/content/ad-glia-fm-prep', 'rev-parse', '--short', 'HEAD'], returncode=0)

**What this shows.** `returncode=0` on the printed `CompletedProcess` means the clone/pull and the `rev-parse` both succeeded — the repo is present at `/content/ad-glia-fm-prep` and added to `sys.path`, so `from src.* import ...` will resolve. The exact commit this session ran on is captured in 3b below (`git_commit` field), which is the value that actually matters for reproducibility.

### 2b — Install requirements_integration.txt

Installs the integration env (scvi-tools 1.4.3, scanpy 1.10.3, scrublet, scib-metrics, harmonypy). Asserts the runtime is Python 3.12 — required by scvi-tools 1.4.x. `torch` / `jax` are TBD-stubs in the pin file; Colab's CUDA-matched wheel resolves them, captured in section 3.

In [7]:
import sys

assert sys.version_info[:2] == (3, 12), (
    f"Expected Python 3.12, got {sys.version_info[:2]}. "
    "Switch the Colab runtime to one that ships 3.12 before continuing."
)

# Pin numpy first so pip picks numpy-1.x-compatible wheels for pandas, scipy, etc.
# Without this, pip resolves against Colab's pre-installed numpy 2.x, then
# downgrades numpy — leaving a binary ABI mismatch that crashes on import.
!pip install numpy==1.26.4
!pip install -r {REPO_PATH}/requirements_integration.txt

**What this shows.** The Python 3.12 assertion passed (no `AssertionError`), and the install completed. Almost every line reads `Requirement already satisfied` because this is the re-run after a kernel restart — the packages were already on disk from the earlier install, so pip resolves rather than re-downloads.

**Why the `numpy==1.26.4` line comes first.** It pins numpy to the 1.x series *before* pip resolves pandas, scipy, and the rest, so those packages get binaries compiled against numpy 1.x. Without it, pip resolves against Colab's pre-installed numpy 2.x, then downgrades numpy at the end — leaving a binary ABI mismatch. The proof the guard worked: cell 5b's `import pandas` runs cleanly here, whereas the first attempt (before the guard) crashed with `ValueError: numpy.dtype size changed`. The dependency-conflict warnings seen on that first install (opencv, shap, rasterio, etc. wanting `numpy>=2`) all involve Colab-default packages this pipeline never imports, so they are harmless noise.

## 3 — Environment capture

Records the *resolved* software stack — pip freeze + Python/CUDA/GPU/OS — to `outputs/software_versions/`. Per [[software-versions-capture]]: the pin file declares what to install (forward-looking); the freeze + env snapshot records what was actually used (backward-looking, paper-ready). Both are committed.

### 3a — pip freeze snapshot

Writes the actually-resolved package set (including transitive deps) to `outputs/software_versions/colab_00_<date>_pip_freeze.txt`.

In [8]:
import os
from datetime import date

NOTEBOOK_ID = "colab_00"
TODAY = date.today().isoformat()

VERSIONS_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(VERSIONS_DIR, exist_ok=True)

FREEZE_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze.txt")
!pip freeze > {FREEZE_PATH}
print(f"Wrote {FREEZE_PATH}")

Wrote /content/ad-glia-fm-prep/outputs/software_versions/colab_00_2026-06-02_pip_freeze.txt


**What this shows.** The fully-resolved package set — every direct dependency plus all transitive ones, at exact versions — is written to `outputs/software_versions/colab_00_2026-06-02_pip_freeze.txt`. This is the backward-looking record ("what actually installed") that pairs with the forward-looking `requirements_integration.txt` ("what we asked for"). Together they let this environment be reconstructed for a paper's Methods section.

### 3b — Python / CUDA / GPU / OS snapshot

Captures what `pip freeze` misses: interpreter version, torch's reported CUDA version, GPU model from `nvidia-smi -L`, OS release, NVIDIA driver. Structured `.json` next to the freeze file. This is what a paper's Methods section is reconstructed from.

In [9]:
import json, platform, subprocess, sys

env_snapshot = {
    "notebook_id": NOTEBOOK_ID,
    "date": TODAY,
    "python_version": sys.version,
    "platform": platform.platform(),
    "os_release": platform.release(),
}

try:
    import torch
    env_snapshot["torch_version"] = torch.__version__
    env_snapshot["torch_cuda_version"] = torch.version.cuda
    env_snapshot["cudnn_version"] = torch.backends.cudnn.version()
    env_snapshot["cuda_available"] = torch.cuda.is_available()
except ImportError:
    env_snapshot["torch_version"] = None

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

env_snapshot["gpu"] = _run(["nvidia-smi", "-L"])
env_snapshot["nvidia_driver"] = _run([
    "nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"
])
env_snapshot["git_commit"] = _run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"])

ENV_JSON_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_env.json")
with open(ENV_JSON_PATH, "w") as f:
    json.dump(env_snapshot, f, indent=2)

print(json.dumps(env_snapshot, indent=2))
print(f"\nWrote {ENV_JSON_PATH}")

{
  "notebook_id": "colab_00",
  "date": "2026-06-02",
  "python_version": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "os_release": "6.6.122+",
  "torch_version": "2.11.0+cpu",
  "torch_cuda_version": null,
  "cudnn_version": null,
  "cuda_available": false,
  "gpu": null,
  "nvidia_driver": null,
  "git_commit": "781148a2e3161084442bc6f798577528e002072b"
}

Wrote /content/ad-glia-fm-prep/outputs/software_versions/colab_00_2026-06-02_env.json


**What this shows.** The hardware + interpreter snapshot for this session:

- **`python_version: 3.12.13`** — confirms the integration-env runtime. scvi-tools 1.4.x requires Python 3.12, so this is the correct interpreter for QC and integration work.
- **`torch_version: 2.11.0+cpu`, `cuda_available: false`, `gpu: null`** — this is a **CPU-only** runtime. That is correct for colab_00: nothing here touches a GPU. Note for later: the integration notebooks (scVI) run faster on a GPU but tolerate CPU; the FM notebooks (Geneformer / scGPT continued-pretraining) **require** switching to a GPU runtime, where torch will reinstall with a CUDA build and these fields will populate.
- **`git_commit: 781148a...`** — the provenance stamp. This commit already contains the numpy-first install fix, so the snapshot faithfully reproduces what ran. It sits one commit behind the current repo HEAD, but the only later changes were a markdown wording tweak and committing these artifact files — neither alters the computation.

## 4 — Audit F: storage budget

Per `docs/setup_audits.md`, populate the storage budget table **before** any raw download. Drive has ca. 80–90 GB usable; target estimated total <70 GB to leave 10 GB+ headroom. Estimates here are placeholders — fill in from each data source's documented file sizes (Synapse / Zenodo / GEO listing pages).

### 4a — Build storage budget and write to audit_report.json

Replace `None` values with verified estimates per source before running `colab_01`. The audit reports `incomplete` while any row is `None`.

In [10]:
import json, os

AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
os.makedirs(os.path.dirname(AUDIT_REPORT_PATH), exist_ok=True)

audit_f = {
    "audit": "F — storage budget",
    "budget_gb": 70,
    "items": [
        {"item": "SEA-AD raw (Allen portal, MTG + DLPFC h5ads)", "est_gb": 13.0, "keep": False, "note": "MTG QC'd h5ad ~5.9 GB confirmed; DLPFC ~6-7 GB est. — delete after per-study processed AnnData saved"},
        {"item": "Haney 2024 raw (GEO GSE254205)",               "est_gb":  0.7, "keep": False, "note": "GSE254205_ad_raw.h5ad.gz — delete after per-study processed AnnData saved"},
        {"item": "Li 2025 raw (GEO GSE237718)",                  "est_gb":  4.7, "keep": False, "note": "GSE237718_RAW.tar — delete after per-study processed AnnData saved"},
        {"item": "Per-study processed AnnData (×3, sparse CSR)", "est_gb":  3.5, "keep": True,  "note": "est. ~2 GB SEA-AD + ~1 GB Li 2025 + ~0.5 GB Haney, post-QC sparse"},
        {"item": "Integrated AnnData (sparse CSR)",              "est_gb":  3.0, "keep": True,  "note": "keep"},
        {"item": "Astro+microglia subset (sparse CSR)",          "est_gb":  0.5, "keep": True,  "note": "keep"},
        {"item": "FM weights (Geneformer + scGPT cached)",       "est_gb":  2.0, "keep": True,  "note": "cached on Colab runtime; Drive footprint minimal if runtime cache persists"},
        {"item": "LoRA adapters (~30MB × 24 at N=3)",            "est_gb":  1.0, "keep": True,  "note": "adapters-only, not full weights"},
        {"item": "Test-set embeddings (~50–100MB × ~30)",        "est_gb":  3.0, "keep": True,  "note": "test-set only by default"},
        {"item": "Logs + checkpoints",                           "est_gb":  0.5, "keep": False, "note": "rotate"},
    ],
}

missing = [r["item"] for r in audit_f["items"] if r["est_gb"] is None]
if missing:
    audit_f["status"] = "incomplete"
    audit_f["note"] = f"{len(missing)} rows TBD — fill before colab_01 download."
    audit_f["missing_rows"] = missing
else:
    total = sum(r["est_gb"] for r in audit_f["items"])
    audit_f["estimated_total_gb"] = total
    audit_f["status"] = "pass" if total < audit_f["budget_gb"] else "fail"

if os.path.exists(AUDIT_REPORT_PATH):
    with open(AUDIT_REPORT_PATH) as f:
        report = json.load(f)
else:
    report = {}
report["audit_f_storage"] = audit_f
with open(AUDIT_REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(audit_f, indent=2))

{
  "audit": "F \u2014 storage budget",
  "budget_gb": 70,
  "items": [
    {
      "item": "SEA-AD raw (Allen portal, MTG + DLPFC h5ads)",
      "est_gb": 13.0,
      "keep": false,
      "note": "MTG QC'd h5ad ~5.9 GB confirmed; DLPFC ~6-7 GB est. \u2014 delete after per-study processed AnnData saved"
    },
    {
      "item": "Haney 2024 raw (GEO GSE254205)",
      "est_gb": 0.7,
      "keep": false,
      "note": "GSE254205_ad_raw.h5ad.gz \u2014 delete after per-study processed AnnData saved"
    },
    {
      "item": "Li 2025 raw (GEO GSE237718)",
      "est_gb": 4.7,
      "keep": false,
      "note": "GSE237718_RAW.tar \u2014 delete after per-study processed AnnData saved"
    },
    {
      "item": "Per-study processed AnnData (\u00d73, sparse CSR)",
      "est_gb": 3.5,
      "keep": true,
      "note": "est. ~2 GB SEA-AD + ~1 GB Li 2025 + ~0.5 GB Haney, post-QC sparse"
    },
    {
      "item": "Integrated AnnData (sparse CSR)",
      "est_gb": 3.0,
      "keep": true,
   

**What this shows.** `status: pass` — the estimated full-pipeline footprint is **31.9 GB** against a **70 GB** budget, leaving ca. 38 GB headroom. Two things to read carefully:

- **31.9 GB is the peak, not the resting size.** It assumes every row coexists — including the three raw downloads (SEA-AD 13 + Li 4.7 + Haney 0.7 GB) sitting on disk at the same time as their processed outputs. The `keep: false` rows are scheduled for deletion: once each study's processed (sparse) AnnData is saved, its raw download is removed. After that cleanup the resting footprint drops to ca. **13 GB** (the `keep: true` rows).
- **The pass is conditional on honoring that deletion discipline.** If raw downloads are not deleted as processing proceeds, the peak climbs and the headroom shrinks.

The numbers are estimates, not measured sizes — e.g. the SEA-AD DLPFC h5ad (ca. 6–7 GB) is a guess until downloaded. Revisit this table with real file sizes at the colab_02 download step.

## 5 — Audit B preemptive: donor-level APOE metadata

Confirms APOE genotype column presence in each study's **donor metadata** (sample-level, typically <10 MB) before committing to the bulk count-matrix download. Expected per the data source lock:

- SEA-AD: 84 donors, APOE present
- Haney 2024: 15 donors, APOE3/3 and APOE4/4 by design (GEO GSE254205)
- Li 2025: 56 donors, purpose-built E2/E3-3/E4 carrier matching (GEO GSE237718)

Don't guess URLs — paste the verified metadata-file path from each source's listing page into the constants in cell 5a. Cell 5b loads, checks for an APOE column, and writes pass/fail to `audit_report.json`.

### 5a — Set per-study donor-metadata paths

Each entry should be a URL or Drive path to a small sample/donor manifest file (CSV/TSV). Leave as `None` to skip that study and have Audit B report `skipped` for it.

In [11]:
# Li 2025 (GSE237718): GEO suppl = GSE237718_RAW.tar only — no standalone donor manifest.
# Haney 2024 (GSE254205): GEO suppl = GSE254205_ad_raw.h5ad.gz only — no standalone manifest.
# APOE metadata for both is embedded in the h5ad obs; checked in per-study QC notebooks.
#
# SEA-AD: Allen portal / AWS open access. Donor-level clinical metadata available;
# paste the URL below once confirmed (small CSV, not the full h5ad).
STUDY_METADATA = {
    "SEA-AD":      {"path": None, "source": "Allen portal / AWS open access",  "expected_donors": 84},
    "Haney 2024":  {"path": None, "source": "GEO GSE254205 — no standalone manifest; APOE in h5ad obs", "expected_donors": 15},
    "Li 2025":     {"path": None, "source": "GEO GSE237718 — no standalone manifest; APOE in h5ad obs", "expected_donors": 56},
}

for study, meta in STUDY_METADATA.items():
    status = "set" if meta["path"] else "TBD"
    print(f"{study:12s}  {status:4s}  source: {meta['source']}")


SEA-AD        TBD   source: Allen portal / AWS open access
Haney 2024    TBD   source: GEO GSE254205 — no standalone manifest; APOE in h5ad obs
Li 2025       TBD   source: GEO GSE237718 — no standalone manifest; APOE in h5ad obs


**What this shows.** All three studies print `TBD` — no donor-metadata path is set. This is deliberate, not an oversight: none of the three studies publishes a small standalone donor manifest. For Li 2025 and Haney 2024 the only GEO supplementary files are the count matrices themselves, with APOE genotype embedded in the h5ad `obs`. So the preemptive Audit B has nothing to load here, and the actual APOE check is deferred to each study's QC notebook where the h5ad is opened. SEA-AD's path is left `None` for now — a clinical metadata CSV may exist on the Allen portal; paste its URL if confirmed.

### 5b — Load each manifest, check APOE column, write Audit B

Detects any column whose name contains `apoe` (case-insensitive). Reports n_donors and APOE value distribution per study. Per-study status:

- `pass` — APOE column found, n_donors ≥ expected
- `warn` — APOE column found, n_donors < expected
- `fail` — no APOE column
- `skipped` — path is None

Overall Audit B `pass` requires every non-skipped study to pass.

In [12]:
import json, os
import pandas as pd

def check_apoe(study, meta):
    if meta["path"] is None:
        return {"study": study, "source": meta["source"], "status": "skipped", "note": "path is None"}
    df = pd.read_csv(meta["path"], sep=None, engine="python")
    apoe_col = next((c for c in df.columns if "apoe" in c.lower()), None)
    donor_col = next(
        (c for c in df.columns if c.lower() in {"donor_id", "subject_id", "individualid", "individual_id", "patient_id"}),
        None,
    )
    if donor_col is None:
        return {"study": study, "source": meta["source"], "status": "warn",
                "note": "no donor_id column found — len(df) would be misleading; inspect columns manually"}
    n_donors = df[donor_col].nunique()
    if apoe_col is None:
        status = "fail"
    elif n_donors < meta["expected_donors"]:
        status = "warn"
    else:
        status = "pass"
    return {
        "study": study,
        "source": meta["source"],
        "path": meta["path"],
        "n_donors": int(n_donors),
        "expected_donors": meta["expected_donors"],
        "apoe_column": apoe_col,
        "apoe_value_counts": (df[apoe_col].astype(str).value_counts().to_dict() if apoe_col else None),
        "status": status,
    }

per_study = [check_apoe(s, m) for s, m in STUDY_METADATA.items()]

checked = [r for r in per_study if r["status"] != "skipped"]
if not checked:
    overall = "incomplete"
elif all(r["status"] == "pass" for r in checked):
    overall = "pass"
elif any(r["status"] == "fail" for r in checked):
    overall = "fail"
else:
    overall = "warn"

audit_b = {"audit": "B — donor APOE metadata", "status": overall, "per_study": per_study}

with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)
report["audit_b_apoe_metadata"] = audit_b
with open(AUDIT_REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(audit_b, indent=2))

{
  "audit": "B \u2014 donor APOE metadata",
  "status": "incomplete",
  "per_study": [
    {
      "study": "SEA-AD",
      "source": "Allen portal / AWS open access",
      "status": "skipped",
      "note": "path is None"
    },
    {
      "study": "Haney 2024",
      "source": "GEO GSE254205 \u2014 no standalone manifest; APOE in h5ad obs",
      "status": "skipped",
      "note": "path is None"
    },
    {
      "study": "Li 2025",
      "source": "GEO GSE237718 \u2014 no standalone manifest; APOE in h5ad obs",
      "status": "skipped",
      "note": "path is None"
    }
  ]
}


**What this shows.** `status: incomplete`, with every study `skipped (path is None)`. **`incomplete` is the expected outcome here, not a failure** — the logic reports `incomplete` precisely when no study was checked, which is the case because no manifest path was provided in 5a. It would only read `fail` if a manifest had been loaded and lacked an APOE column.

The real APOE confirmation happens per-study by loading the h5ad and inspecting `obs`: **Li 2025** in colab_01, **SEA-AD** in colab_02, **Haney 2024** in its QC notebook. The genotype designs those checks must confirm are: Li 2025 — purpose-built E2 / E3-3 / E4 carrier matching; Haney 2024 — APOE3/3 vs APOE4/4; SEA-AD — APOE present in donor metadata. Any study whose h5ad turns out to lack APOE would be disqualified, since APOE stratification is the project's core axis.

## 6 — Handoff to colab_01

Prints a final summary and the explicit `git add / commit / push` commands needed to land the env snapshot + audit report on GitHub before the runtime resets. `colab_01_li2025_qc.ipynb` is the next notebook — vertical slice on the smallest study (Li 2025, 343k nuclei, GEO GSE237718) before scaling to SEA-AD + Haney 2024.

### 6a — Summary + commit instructions

Reads back `audit_report.json` and lists artifact paths. Runs the git commands inline — if commit/push fails (auth, network), the printed shell commands are the recovery path.

In [13]:
import json, os, shlex, subprocess

with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)

print("=== Audit summary ===")
for k, v in report.items():
    print(f"  {k}: {v.get('status')}")

print("\n=== Artifacts written ===")
for p in [FREEZE_PATH, ENV_JSON_PATH, AUDIT_REPORT_PATH]:
    print(f"  {p}")

print("\n=== Commit + push (run from a fresh cell if these fail) ===")
rel_paths = [os.path.relpath(p, REPO_PATH) for p in [FREEZE_PATH, ENV_JSON_PATH, AUDIT_REPORT_PATH]]
msg = f"colab_00: env capture + preemptive audits F, B ({TODAY})"
cmds = [
    f"cd {REPO_PATH} && git add {' '.join(rel_paths)}",
    f"cd {REPO_PATH} && git commit -m {__import__('shlex').quote(msg)}",
    f"cd {REPO_PATH} && git push",
]
for c in cmds:
    print(f"  {c}")

print("\nNext: colab_01_li2025_qc.ipynb (Li 2025 download + secondary_qc + Audit A on this study).")

=== Audit summary ===
  audit_f_storage: pass
  audit_b_apoe_metadata: incomplete

=== Artifacts written ===
  /content/ad-glia-fm-prep/outputs/software_versions/colab_00_2026-06-02_pip_freeze.txt
  /content/ad-glia-fm-prep/outputs/software_versions/colab_00_2026-06-02_env.json
  /content/ad-glia-fm-prep/outputs/audit_report.json

=== Commit + push (run from a fresh cell if these fail) ===
  cd /content/ad-glia-fm-prep && git add outputs/software_versions/colab_00_2026-06-02_pip_freeze.txt outputs/software_versions/colab_00_2026-06-02_env.json outputs/audit_report.json
  cd /content/ad-glia-fm-prep && git commit -m 'colab_00: env capture + preemptive audits F, B (2026-06-02)'
  cd /content/ad-glia-fm-prep && git push

Next: colab_01_li2025_qc.ipynb (Li 2025 download + secondary_qc + Audit A on this study).


**What this shows.** colab_00 is complete. The summary reports **`audit_f_storage: pass`** and **`audit_b_apoe_metadata: incomplete`** — the storage budget clears, and the APOE audit is deferred by design (see 5b). Three provenance artifacts were written: the pip-freeze, the env snapshot, and the audit report.

**On the push.** This cell only *prints* the git commands — it does not run them. In practice the push was done manually from the local WSL machine, because the Colab runtime has no GitHub credentials configured (`git push` from Colab fails with `could not read Username`). That auth fragility is exactly why the push is left as a manual, copy-paste step rather than an automated cell.

**Next:** `colab_01_li2025_qc.ipynb` — download Li 2025 (343k nuclei, the smallest of the three studies), run secondary QC, and execute Audit A on it. Starting with the smallest study is a deliberate vertical slice: it shakes out the per-study QC pipeline cheaply before scaling to SEA-AD (1.2M nuclei) and Haney.